In [2]:
import os
import sys
import warnings
import torch
import pandas as pd
import numpy as np
import re

# Stopwords & punctuation
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold

# Embedding models
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models.word2vec import Word2Vec
from gensim.models.doc2vec import TaggedDocument, Doc2Vec

# Classifiers
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.ensemble import RandomForestClassifier

# Élimination des avertissements
if not sys.warnoptions:
    warnings.simplefilter("ignore")
    os.environ["PYTHONWARNINGS"] = "ignore"

In [3]:
punct = string.punctuation.replace('-', '')
stopw = stopwords.words('english') + list(punct)

def tokenize_remove_stop_words(text: str):
    return [token for token in word_tokenize(text) if 
            token.lower() not in stopw and
            len(token) > 2 and  # Mots de moins de 2 lettres
            not (bool(re.search(r'\d', token))) and # Mots contenant des chiffres
            not (any(char in punct for char in token))] # Mots contenant des signes de ponctuation

def vectorize_word2vec(corpus, w2v_model):    
    def vectorize(document_tokenized):
        words_vecs = [w2v_model.wv[word] for word in document_tokenized if word in w2v_model.wv]
        if len(words_vecs) == 0:
            return np.zeros(w2v_model.vector_size)
        words_vecs = np.array(words_vecs)
        return words_vecs.mean(axis=0)
    
    tokenized_corpus = [list(tokenize_remove_stop_words(doc)) for doc in corpus]
    X = np.array([vectorize(doc) for doc in tokenized_corpus])
    return X

**Lecture des données**

In [4]:
# Lecture du jeu de données et séparation de celles-ci en ensembles d'entraînement et de test
train = pd.read_csv('../data/training_datasets/train_dataset_40pc.csv')
test = pd.read_csv('../data/test_dataset_10.csv')

**Entraînement des modèles**

In [5]:
train['category'] = train['category'].apply(lambda x: 1 if x == 'incel' else 0)
test['category'] = test['category'].apply(lambda x: 1 if x == 'incel' else 0)

X_train, y_train = train.text_post.astype('str'), train.category
X_test, y_test = test.text_post.astype('str'), test.category

In [6]:
tokens = [list(tokenize_remove_stop_words(doc)) for doc in X_train]
features_w2v = [100, 200, 300, 400, 500]

# On crée un modèle distinct pour chaque nombre de dimensions à tester (features_w2v)
w2v_models = [
    Word2Vec(
        tokens,
        vector_size=n_features
    ) 
    for n_features in features_w2v
]

In [7]:
word2vec_transformers = [FunctionTransformer(
    lambda x: vectorize_word2vec(
        x, 
        w2v_model=w2v_models[i]
    )
) for i in range(len(w2v_models))]

# # Sentence-bert model
# # Define SBERT embedder
# sbert_vectorizer = FunctionTransformer(
#     lambda x: sbert_model.encode(
#         x.squeeze().astype(str).values,
#         batch_size=64,
#         convert_to_numpy=True,
#         show_progress_bar=True,
#         device=device
#     )
# )

In [8]:
vectorizers = {
    # 'TfIdfVectorizer' : TfidfVectorizer(stop_words=stopw, tokenizer=word_tokenize),
    'w2vVectorizer__100' : word2vec_transformers[0],
    'w2vVectorizer__200' : word2vec_transformers[1],
    'w2vVectorizer__300' : word2vec_transformers[2],
    'w2vVectorizer__400' : word2vec_transformers[3],
    'w2vVectorizer__500' : word2vec_transformers[4],
    # 'S-BERTVectorizer' : sbert_vectorizer,
}

In [9]:
# Définition du pipeline
features_tfidf = [5000, 10000, 15000]

tfidf_param_grid = {
        "vectorizer__max_features": features_tfidf,
        "classify" : [
            LogisticRegression(n_jobs=1, solver='saga'), 
            LinearSVC(dual="auto"),
            MultinomialNB(),
        ]
}

w2v_d2v_param_grid = {
        "classify" : [
            RandomForestClassifier(n_estimators=32, min_samples_split=2), 
            LinearSVC(dual="auto"),
            GaussianNB()
        ]
}

sbert_param_grid = {
    "classify" : [
        LogisticRegression(n_jobs=1, solver='saga'),
        LinearSVC(dual="auto")
    ]
}

param_grid = {
    "TfIdfVectorizer": tfidf_param_grid,
    "w2vVectorizer__100": w2v_d2v_param_grid,
    "w2vVectorizer__200": w2v_d2v_param_grid,
    "w2vVectorizer__300": w2v_d2v_param_grid,
    "w2vVectorizer__400": w2v_d2v_param_grid, 
    "w2vVectorizer__500": w2v_d2v_param_grid,
    "S-BERTVectorizer" : sbert_param_grid    
}

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RepeatedStratifiedKFold, GridSearchCV
from sklearn.ensemble import RandomForestClassifier

# Define the repeated stratified k-fold cross-validator
cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=42)

# To store results
results = {}

# Loop through each vectorizer and corresponding param grid
for vectorizer_name, vectorizer in vectorizers.items():
    # Fetch the parameter grid for this vectorizer
    specific_param_grid = param_grid.get(vectorizer_name, {})

    pipeline = Pipeline([
        ("vectorizer", vectorizer),
        ("classify", "passthrough")  
    ])
    
    # Define the grid search
    grid_search = GridSearchCV(
        pipeline, 
        param_grid=specific_param_grid, 
        verbose=5, 
        cv=cv,
        refit='f1_macro', 
        scoring=['accuracy', 'f1_macro']
    )
    
    # Run the grid search
    print(f"Running GridSearchCV for {vectorizer_name}...")
    
    # Fit the model
    try:
        grid_search.fit(X_train, y_train)  # Replace with your actual training data
    except Exception as e:
        print(f"Error during fitting for {vectorizer_name}: {e}")
        continue
    
    # Store results
    results[vectorizer_name] = {
        "best_params": grid_search.best_params_,
        "best_score": grid_search.best_score_
    }

# Display the results
for vectorizer_name, result in results.items():
    print(f"Results for {vectorizer_name}:")
    print(f"  Best Parameters: {result['best_params']}")
    print(f"  Best Accuracy: {result['best_score']:.4f}")


Running GridSearchCV for w2vVectorizer__100...
Fitting 30 folds for each of 3 candidates, totalling 90 fits


In [ ]:
result

NameError: name 'results' is not defined

In [ ]:
# cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=10, random_state=0)

# grid_search = GridSearchCV(
#     pipeline, 
#     param_grid=param_grid, 
#     verbose=5, 
#     cv=cv,
#     refit='f1_macro', 
#     scoring=['accuracy','f1_macro']
# )

In [ ]:
# grid_search.fit(X_train, y_train)

In [ ]:
# grid_search.best_params_

In [ ]:
# results_cv = pd.DataFrame(grid_search.cv_results_)
# results_cv